## Script to calculte the map demand to node network
### Using the PtX Markets demand results for industry and transport

In [1]:
import os 
import geopandas as gpd
import pandas as pd
from shapely import wkt
import geopy.distance
from shapely.geometry import Point
from tqdm import tqdm
from datetime import date
import matplotlib.pyplot as plt
import numpy as np

# To fasten the find closest location function 
from sklearn.neighbors import BallTree

# For Voronoi polygons implementation
from scipy.spatial import Voronoi
from shapely.geometry import Polygon, MultiPoint

In [2]:
#Setting for Script 
Save_Output = False
Visualisation = True

In [3]:
#Binary Scenario Settings
# Categories from results
Steel = True 
Chemicals = False 
Non_Metallic_Minerals = True 
Passenger_Road = True 
Passenger_Train = False 
Passenger_Air = False

Freight_Road = True 
Freight_Rail = False	
Maritime = True

# Maybe add aggregates for the final visualization and choices 
# Industry = True 
# Passenger = True 
# Freight = True 

##### Functions to run the main script

In [4]:
# Function for WKT points to clean the data
def tuple_to_wkt_point(s):
    x_str, y_str = s.strip().replace("(", "").replace(")", "").split(",")
    return f"POINT({x_str} {y_str})"

In [5]:
# Function to map demand supply and storage to closest nodes 
def AggregatedDemand(df1, df2):
    # Ensure Demand column is initialized with zeros 
    if 'Demand' not in df1.columns:
        df1['Demand'] = 0.0
    else:
        df1['Demand'] = df1['Demand'].fillna(0.0)

    for index, row1 in tqdm(df1.iterrows(), total=df1.shape[0]):
        for _, row2 in df2.iterrows():
            if row1['ID'] == row2['Closest_node']:
                df1.at[index, 'Demand'] += row2['Peak Load [MWh/h]']
    return df1


def AggregatedSupply(df1, df2):
    for index, row1 in tqdm(df1.iterrows(), total=df1.shape[0]):
        for _, row2 in df2.iterrows():
            if row1['ID'] == row2['Closest_node']:
                df1.loc[index, 'Supply'] += row2['Peak Load [MWh/h]']
    return df1

def AddStorage(df1, df2, aggregated):
    if sum(df2['Peak Load [MWh/h]']) <= 0:
        df2['Peak Load [MWh/h]'] = df2['Peak Load [MWh/h]'] * (-1)
        aggregated = AggregatedSupply(df1, df2)
    else:
        aggregated = AggregatedDemand(df1, df2)

    return aggregated

In [6]:
# Clean the data and plz file 
def clean_column(df, column_name):
    df[column_name] = (
        df[column_name].astype(str)
        .str.replace(',', '.', regex=False)
        .str.replace(' ', '', regex=False)
    )
    df[column_name] = pd.to_numeric(df[column_name], errors='coerce')
    return df

def clean_plz(series):
    return (
        series.astype(str)
              .str.replace('\u00A0', '', regex=False)  # need to remove breaks or space and have the normal plz format
              .str.replace(' ', '', regex=False)     
              .str.strip()
              .str.zfill(5)  
    )

In [7]:
def find_closest_location(demand_gdf, nodes_gdf):
    # Convert lat/lon to radians 8to use BallTree)
    demand_coords = np.radians(demand_gdf[['Latitude', 'Longitude']].values)
    node_coords = np.radians(nodes_gdf[['Latitude', 'Longitude']].values)
    tree = BallTree(node_coords, metric='haversine')

    # Faster query of closest node for each demand point
    dist, ind = tree.query(demand_coords, k=1)
    dist_km = dist[:, 0] * 6371  # Conversion from radians to kilometers
    node_ids = nodes_gdf.iloc[ind[:, 0]]['ID'].values
    return node_ids, dist_km

In [8]:
# In the case of sheets with direct access to latitude and longitude 
def latlon_case(df):
    for col in ['Latitude', 'Longitude', 'Peak Load [MWh/h]']:
        df = clean_column(df, col)

    df = df.dropna(subset=['Latitude', 'Longitude'])
    geometry = gpd.points_from_xy(df['Longitude'], df['Latitude'])
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

    return gdf[['Latitude', 'Longitude', 'Peak Load [MWh/h]', 'geometry']]


In [9]:
# Function for the Voronoi polygons and NUTs3 region case 
def allocate_demand_with_voronoi(df, nuts3_gdf, country_boundary=None):
    # Merge with geometries
    df = df.merge(nuts3_gdf[['NUTS_ID', 'geometry']], left_on='NUTS3', right_on='NUTS_ID', how='left')

    # Create centroids to generate Voronoi
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326').to_crs('EPSG:3857')
    points = np.array([[geom.centroid.x, geom.centroid.y] for geom in gdf.geometry])
    vor = Voronoi(points)

    # Build Voronoi polygons
    polygons = []
    for region_index in vor.point_region:
        region = vor.regions[region_index]
        if not region or -1 in region:
            polygons.append(None)
        else:
            poly = Polygon([vor.vertices[i] for i in region])
            polygons.append(poly)

    # Create Voronoi GeoDataFrame
    voronoi_gdf = gpd.GeoDataFrame(gdf.copy(), geometry=polygons, crs='EPSG:3857')
    voronoi_gdf = voronoi_gdf.dropna(subset=["geometry"])

    # Compute area-based weights and allocate demand
    voronoi_gdf['area'] = voronoi_gdf.geometry.area
    total_area_by_nuts = voronoi_gdf.groupby('NUTS3')['area'].transform('sum')
    voronoi_gdf['demand_weight'] = voronoi_gdf['area'] / total_area_by_nuts
    voronoi_gdf['allocated_demand'] = voronoi_gdf['demand_weight'] * voronoi_gdf['Peak Load [MWh/h]']

    # Output points for next steps
    voronoi_gdf['geometry'] = voronoi_gdf.geometry.centroid
    voronoi_gdf = voronoi_gdf.to_crs('EPSG:4326')
    voronoi_gdf['Latitude'] = voronoi_gdf.geometry.y
    voronoi_gdf['Longitude'] = voronoi_gdf.geometry.x

    return voronoi_gdf[['NUTS3', 'Latitude', 'Longitude', 'allocated_demand', 'geometry']] \
        .rename(columns={'allocated_demand': 'Peak Load [MWh/h]'})

In [10]:
# Read the sectors and create dataframe with precise geometries for further mapping demand
def read_sector(full_path, sheet, plz_gdf, nuts3_gdf):
    df = pd.read_excel(full_path, sheet_name=sheet)

    if {'Latitude', 'Longitude', 'Peak Load [MWh/h]'}.issubset(df.columns):
        return latlon_case(df)

    elif 'Standort-PLZ' in df.columns:
        # Merge the PLZ code with geodataframe creted earlier 
        df['Standort-PLZ'] = clean_plz(df['Standort-PLZ'])
        df = df.merge(plz_gdf[['Postleitzahl / Post code', 'geometry']], left_on='Standort-PLZ', right_on='Postleitzahl / Post code', how='left')
        df = df.dropna(subset=['geometry'])

        # Create the geodataframe with correct geometry 
        gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
        gdf['geometry'] = gdf['geometry'].apply(lambda geom: geom.centroid if geom.geom_type != 'Point' else geom)
        return gdf

    elif 'NUTS3' in df.columns:
        # Handle NUTS3 case using Voronoi polygons
        return allocate_demand_with_voronoi(df, nuts3_gdf)

    else:
        print(f"Sheet '{sheet}' has no recognizable location format.")
        return gpd.GeoDataFrame()

##### Call files and initialize output 

In [11]:
# Define the scenario demand settings
# File name of the scenario input file and output file of aggregated
Date = str(date.today())
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw')
input_file = '\\PtX_demand_DE.xlsx'  
full_input_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path + input_file))

topo_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
topo_file = '\\input_network_data.xlsx'
full_topo_path = os.path.abspath(os.path.join(os.getcwd(), topo_file_path + topo_file))

gdp_file_path = os.path.join('..', '..','01_data', '01_input_data', '01_raw', 'NUTS_eurostat')
gdp_file = '\\GDP_NUTS3.csv'
full_gdp_path = os.path.abspath(os.path.join(os.getcwd(), gdp_file_path + gdp_file))

output_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
output_file = '\\Demand_Nodes_'+Date+'.csv'  
full_output_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path + output_file))


In [12]:
topo_file = pd.read_excel(full_topo_path)
source_df = topo_file[['source_name', 'source']].drop_duplicates().rename(columns={'source_name':'node_id', 'source':'geometry'})
target_df = topo_file[['target_name', 'target']].drop_duplicates().rename(columns={'target_name':'node_id', 'target':'geometry'})
nodes = pd.concat([source_df, target_df], ignore_index=True)

nodes['geometry'] = (nodes['geometry'].astype(str).apply(tuple_to_wkt_point).apply(wkt.loads))

nodes_gdf = gpd.GeoDataFrame(nodes, crs='EPSG:3857')
nodes_gdf.to_crs('EPSG:4326', inplace = True)
nodes_gdf['Longitude'] = nodes_gdf.geometry.apply(lambda p: p.x)
nodes_gdf['Latitude'] = nodes_gdf.geometry.apply(lambda p: p.y)
nodes_gdf['Supply'] = 0
nodes_gdf['Demand'] = 0 

In [13]:
input_file_path = os.path.join('..', '..', '01_data', '01_input_data', '01_raw', '02_Industrial_Site_Database.xlsx')
industrial_sites = pd.read_excel(input_file_path, sheet_name='Database')

industrial_sites = industrial_sites[industrial_sites['Country'] == 'DE']
#print(industrial_sites.head(10))

In [14]:
industrial_sites = industrial_sites.rename(columns={
    'Process status qup': 'Process',
    'Latitude': 'lat',
    'Longitude': 'lon',
    'Production in tons (calibrated)': 'production_tons'
})

cols = ['id_site', 'Site', 'Product', 'Process', 'lat', 'lon', 'production_tons']
german_sites = industrial_sites[industrial_sites['Country'] == 'DE'][cols].copy()

german_sites = gpd.GeoDataFrame(
    german_sites,
    geometry=gpd.points_from_xy(german_sites['lon'], german_sites['lat']),
    crs='EPSG:4326'
)

print(german_sites.head())

    id_site                                  Site         Product  \
42    116.0  ROGESA Roheisengesellschaft Saar mbH  Steel, primary   
43    116.0  ROGESA Roheisengesellschaft Saar mbH  Steel, primary   
44    119.0                 Raffinerie Heide GmbH        Ethylene   
45    120.0                 YARA Brunsbüttel GmbH         Ammonia   
46    124.0            ArcelorMittal Hamburg GmbH  Steel, primary   

                Process        lat       lon production_tons  \
42        Blast furnace  49.352137  6.743653     1740495.613   
43        Blast furnace  49.352137  6.743653     2056949.361   
44       Steam cracking  54.160831  9.079774       86958.527   
45          Ammonia SMR  53.910644  9.207871     559759.7598   
46  Direct reduction NG  53.524785  9.900051     474680.6218   

                    geometry  
42  POINT (6.74365 49.35214)  
43  POINT (6.74365 49.35214)  
44  POINT (9.07977 54.16083)  
45  POINT (9.20787 53.91064)  
46  POINT (9.90005 53.52478)  


In [20]:
# Read the PtX data
target_year = 2030 
ptx_df = pd.read_excel(full_input_path, decimal=',', thousands='.')
H2_demand = ptx_df[(ptx_df['FuelGroup'] == 'Hydrogen') & (ptx_df['Year'] == target_year)].iloc[0]

# Define the columns (sectors) and extract the national demand in EJ
H2_demand = ptx_df[(ptx_df['FuelGroup'] == 'Hydrogen') & (ptx_df['Year'] == target_year)].iloc[0]

# Drop FuelGroup and Year to keep only relevant columns
national_H2_demand_EJ = H2_demand.drop(['FuelGroup', 'Year']).dropna()

print(f"National H2 Demand (EJ) for {target_year}:\n{national_H2_demand_EJ}")

National H2 Demand (EJ) for 2030:
Iron & steel     0.017652
Pass Road        0.005571
Pass Aviation         0.0
Freight Road     0.003325
Name: 1, dtype: object


In [25]:
# Define parameters
industrial_sectors_ptx = ['Iron & steel', 'Non-metallic minerals', 'Chemicals'] 
all_demand_points = []
nodes_gdf.rename(columns={'node_id': 'ID'}, inplace=True, errors='ignore')

# Mapping PtX categories to specific products/subsectors in your site database.
PTX_TO_SITE_PRODUCT_MAP = {
    'Iron & steel': ['Steel', 'Iron'],
    'Chemicals': ['Ammonia', 'Ethylene', 'Methanol'],
    'Non-metallic minerals': ['Cement', 'Lime', 'Glass'] # Missing in our database
}

# Loop through industrial sectors and allocate demand
for sector in industrial_sectors_ptx:
    national_demand_EJ = national_H2_demand_EJ.get(sector, 0.0)
    
    if national_demand_EJ > 0:
        print(f"Allocating industrial demand for: {sector} ({national_demand_EJ:.4f} EJ)")
        
        # 1. Filter sites based on the specific products/subsectors
        target_products = PTX_TO_SITE_PRODUCT_MAP.get(sector, [])
        
        if not target_products:
             print(f"  Warning: No product mapping defined for {sector}. Skipping.")
             continue

        # Create a boolean mask: True if 'Product' contains ANY of the target strings
        product_mask = german_sites['Product'].apply(lambda p: any(tp in p for tp in target_products))
        site_gdf_sector = german_sites[product_mask].copy()
        
        # 2. Use 'production_tons' as the weight
        total_tons = site_gdf_sector['production_tons'].sum()
        
        if total_tons > 0:
            # Calculate site-specific demand
            site_gdf_sector['Demand_Share'] = site_gdf_sector['production_tons'] / total_tons
            site_gdf_sector['Annual_Demand_EJ'] = site_gdf_sector['Demand_Share'] * national_demand_EJ
            
            # Prepare for closest node search (requires Latitude/Longitude)
            site_gdf_sector.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'}, inplace=True)
            
            # Find closest node
            site_gdf_sector['Closest_node'], site_gdf_sector['Distance_km'] = find_closest_location(site_gdf_sector, nodes_gdf)
            
            # Store the final disaggregated demand points
            demand_to_aggregate = site_gdf_sector[site_gdf_sector['Annual_Demand_EJ'] > 0][['Closest_node', 'Annual_Demand_EJ']]
            all_demand_points.append(demand_to_aggregate)
            
            print(f"  --> Allocated to {len(demand_to_aggregate)} site points ({site_gdf_sector['Site'].nunique()} unique plants).")
        else:
            print(f"  Warning: No production data found for {sector} using products {target_products}. Skipping.")
    else:
        print(f"  {sector} demand is 0 or missing.")

Allocating industrial demand for: Iron & steel (0.0177 EJ)
  --> Allocated to 16 site points (8 unique plants).
  Non-metallic minerals demand is 0 or missing.
  Chemicals demand is 0 or missing.


In [ ]:
# Ensure all_demand_points has content from the industrial allocation
if all_demand_points:
    # Concatenate all lists of demand points into one DataFrame
    all_demand_points_df = pd.concat(all_demand_points, ignore_index=True)
    aggregated_demand = all_demand_points_df.groupby('Closest_node').agg({'Annual_Demand_EJ': 'sum'}).reset_index()
    aggregated_demand.rename(columns={'Closest_node': 'ID', 'Annual_Demand_EJ': 'Demand_EJ'}, inplace=True)

    # 3. Perform the merge/join
    nodes_gdf['Demand_EJ'] = 0.0 
    nodes_gdf = nodes_gdf.merge(aggregated_demand, on='ID', how='left', suffixes=('_old', '_new'))
    nodes_gdf['Demand_EJ'] = nodes_gdf['Demand_EJ_new'].fillna(0.0) 
    
    # Clean up the merged column
    nodes_gdf.drop(columns=['Demand_EJ_new'], inplace=True, errors='ignore')
    nodes_gdf['Demand'] = 0.0 # Keep this placeholder MWh/h column

    print("Demand aggregation successful.")
    print(f"Total H2 Demand aggregated: {nodes_gdf['Demand_EJ'].sum():.4f} EJ")

else:
    # If all_demand_points is empty, we must still create the column for the visualization!
    nodes_gdf['Demand_EJ'] = 0.0
    nodes_gdf['Demand'] = 0.0
    print("Warning: No valid demand points were generated. Setting Demand_EJ to zero across all nodes.")


--- Aggregating Demand to Network Nodes ---
Demand aggregation successful.
Total H2 Demand aggregated: 0.0212 EJ


C:\Users\mar.eco\AppData\Local\Temp\ipykernel_7392\1349223068.py:24: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  nodes_gdf['Demand_EJ'] = nodes_gdf['Demand_EJ_new'].fillna(0.0)


In [26]:
if Visualisation:
    import plotly.graph_objects as go
    import numpy as np
    
    # 1. Use the new Demand_EJ column for plotting
    df_plot = pd.DataFrame({
        'ID': nodes_gdf['ID'],
        'Longitude': nodes_gdf['Longitude'],
        'Latitude': nodes_gdf['Latitude'],
        'Demand': nodes_gdf['Demand_EJ'] # *** CHANGED TO Demand_EJ ***
    })

    # Filter out nodes with zero demand to focus the plot (optional, but cleaner)
    df_plot_active = df_plot[df_plot['Demand'] > 0].copy()
    df_plot_inactive = df_plot[df_plot['Demand'] == 0].copy()
    
    # need to scale the demand for the points size on map 
    # np.log1p(d) provides a good visual scaling for demand data
    df_plot_active['ScaledDemand'] = df_plot_active['Demand'].apply(lambda d: np.log1p(d)) + 0.1
    # df_plot['Used'] is now implicitly handled by df_plot_active
    
    # Plot the map to visualize
    fig = go.Figure()

    # Add ALL nodes (as small blue dots for context)
    fig.add_trace(go.Scattergeo(
        lon=df_plot_inactive['Longitude'],
        lat=df_plot_inactive['Latitude'],
        mode='markers',
        marker=dict(
            size=2, # Small size for unused nodes
            color='blue',
            line=dict(width=0.5, color='darkblue'),
            opacity=0.5
        ),
        name='Unused Nodes',
        hoverinfo='skip'
    ))

    # Add ACTIVE demand nodes (colored red, scaled by demand)
    for _, row in df_plot_active.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[row['Longitude']],
            lat=[row['Latitude']],
            mode='markers',
            marker=dict(
                # Ensure a minimum size so points are visible
                size=max(row['ScaledDemand']*100, 5), # Increased scaling factor (100) for better visibility of EJ units
                color='red', 
                line=dict(width=0.5, color='black'),
                opacity=0.7
            ),
            # Update hovertext to show the correct unit (EJ)
            hovertext=f"ID: {row['ID']}<br>Demand: {row['Demand']:.4f} EJ", 
            name='H2 Demand (EJ)',
            showlegend=False
        ))
    
    # Add a title specific to the current allocation state
    fig.update_layout(
        title_text=f'Aggregated H2 Demand (Iron & Steel Only) for {target_year} [EJ]',
        geo=dict(
            scope='europe',
            projection_type='natural earth',
            center=dict(lat=51.2, lon=10.4),
            lataxis=dict(range=[47, 55]), # Focus on Germany
            lonaxis=dict(range=[5, 16]),  # Focus on Germany
            resolution=50
        ),
        margin={"r":0,"t":50,"l":0,"b":0},
        width=900,
        height=700,
    )

    fig.show()

In [ ]:
'''# Read Eurostat data for GDP per NUTS3 regions
gdp = pd.read_csv(full_gdp_path, sep=',', decimal=',', encoding='utf-8')
gdp_nuts3 = gdp[gdp['geo'].str.len() == 5].copy()

# Filer Germany and only important columns
germany_gdp = gdp_nuts3[gdp_nuts3['geo'].str.startswith('DE')].copy()
cols = ['geo', 'Geopolitical entity (reporting)', 'TIME_PERIOD', 'OBS_VALUE']
germany_gdp = germany_gdp[cols]
germany_gdp_22 = germany_gdp[germany_gdp['TIME_PERIOD'] == 2022]

germany_gdp_unique = germany_gdp_22.drop_duplicates(subset='geo')
print(germany_gdp_unique)

total_gdp = germany_gdp_unique['OBS_VALUE'].sum()
germany_gdp_unique['GDP_share'] = germany_gdp_unique['OBS_VALUE'] / total_gdp
print(germany_gdp_unique[['geo', 'Geopolitical entity (reporting)', 'OBS_VALUE', 'GDP_share']].head())'''

C:\Users\mar.eco\AppData\Local\Temp\ipykernel_9768\1662886834.py:2: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  gdp = pd.read_csv(full_gdp_path, sep=',', decimal=',', encoding='utf-8')


        geo Geopolitical entity (reporting)  TIME_PERIOD OBS_VALUE
2069  DE111           Stuttgart, Stadtkreis         2022     94600
2078  DE112                       Böblingen         2022     76200
2087  DE113                       Esslingen         2022     51400
2096  DE114                       Göppingen         2022     35100
2105  DE115                     Ludwigsburg         2022     48700
...     ...                             ...          ...       ...
6144  DEG0R                   Wartburgkreis         2022     37100
6153  DEG0S          Suhl, Kreisfreie Stadt         2022     35500
6162  DEG0T                       Ilm-Kreis         2022     35300
6171  DEG0U             Saalfeld-Rudolstadt         2022     31900
6180  DEG0V                       Sonneberg         2022     29700

[400 rows x 4 columns]
        geo Geopolitical entity (reporting) OBS_VALUE GDP_share
2069  DE111           Stuttgart, Stadtkreis     94600  0.005407
2078  DE112                       Böblingen 

C:\Users\mar.eco\AppData\Local\Temp\ipykernel_9768\1662886834.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  germany_gdp_unique['GDP_share'] = germany_gdp_unique['OBS_VALUE'] / total_gdp


#### PLZ Shapefile
Downloaded from Opendatasoft:  
https://public.opendatasoft.com/explore/dataset/georef-germany-postleitzahl/export/

- Load the file (GeoJSON, csv or Excel)
- Check geometry type and postal codes
- Identify any missing plz
- Fill in the missing geometries if possible

In [14]:
# Load the PLZ shapefile and geometry
geojson_path = '../../01_data/01_input_data/01_raw/plz_shape_file/georef-germany-postleitzahl.geojson'
excel_path = '../../01_data/01_input_data/01_raw/plz_shape_file/georef-germany-postleitzahl.xlsx'
plz_df = pd.read_excel(excel_path)

# Clean the PLZ codes 
plz_df['Postleitzahl / Post code'] = clean_plz(plz_df['Postleitzahl / Post code'])

# Convert the DataFrame to a GeoDataFrame
plz_gdf = gpd.GeoDataFrame(plz_df, geometry=gpd.points_from_xy(plz_df['geo_point_2d'].str.extract(r'(\d+\.\d+), (\d+\.\d+)')[1].astype(float), plz_df['geo_point_2d'].str.extract(r'(\d+\.\d+), (\d+\.\d+)')[0].astype(float)), crs='EPSG:4326')

##### Sources for Missing PLZ Codes
For the missing PLZ codes, the following sources were used to find the necessary coordinates:

- **12861 (Berlin-Marzahn)** from Wikipedia
- **51368 (Leverkusen)**: data from [PLZ-Guru](https://www.plz-guru.de/plz/51368)
- **63784 (Obernburg am Main)**: data from [Latitude and Longitude Finder](https://latitudelongitude.org/de/obernburg-am-main/#google_vignette)
- **67056 (Ludwigshafen am Rhein)**: similarly from [Latitude and Longitude Finder](https://latitudelongitude.org/de/ludwigshafen-am-rhein/?utm_source=chatgpt.com)
- **71059 (Sindelfingen)**: from Wikipedia
- **01956 (Senftenberg)**: from [Free Country Maps](https://www.freecountrymaps.com/map/towns/germany/1676297307/)
- **28023 (Bremen)**: from github German plz code sources
- **63659 (Stockheim, Glauburg)**: from GitHub sources
- **67102 (Zeitz)**: from Wikipedia
- **14627 (Elstal Wustermark)**: from GitHub sources

In [16]:
# Data for missing PLZ codes
missing_plz_data = [
    {'Postleitzahl / Post code': '12861', 'city': 'Berlin-Marzahn', 'geo_point_2d': '52.5500, 13.5500'},
    {'Postleitzahl / Post code': '51368', 'city': 'Leverkusen', 'geo_point_2d': '51.0377, 6.9865'},
    {'Postleitzahl / Post code': '63784', 'city': 'Obernburg am Main', 'geo_point_2d': '49.83577, 9.13101'},
    {'Postleitzahl / Post code': '67056', 'city': 'Ludwigshafen am Rhein', 'geo_point_2d': '49.48121, 8.44641'},
    {'Postleitzahl / Post code': '71059', 'city': 'Sindelfingen', 'geo_point_2d': '48.70746, 9.00441'},
    {'Postleitzahl / Post code': '01956', 'city': 'Senftenberg', 'geo_point_2d': '51.5192, 14.0047'},
    {'Postleitzahl / Post code': '28023', 'city': 'Bremen', 'geo_point_2d': '53.0736, 8.8064'},
    {'Postleitzahl / Post code': '63659', 'city': 'Stockheim (Glauburg)', 'geo_point_2d': '50.3246, 9.0176'},
    {'Postleitzahl / Post code': '67102', 'city': 'Zeitz', 'geo_point_2d': '51.0478, 12.1383'},
    {'Postleitzahl / Post code': '14627', 'city': 'Elstal (Wustermark)','geo_point_2d': '52.5400, 12.9900'}
]

In [17]:
# Create a GeoDataFrame like above 
missing_plz_df = pd.DataFrame(missing_plz_data)
missing_plz_df['Postleitzahl / Post code'] = clean_plz(missing_plz_df['Postleitzahl / Post code'])
missing_plz_df['geometry'] = missing_plz_df['geo_point_2d'].apply(lambda x: Point(float(x.split(', ')[1]), float(x.split(', ')[0])))
missing_plz_gdf = gpd.GeoDataFrame(missing_plz_df, geometry='geometry', crs='EPSG:4326')

In [18]:
# Merge with the existing plz shape file
plz_gdf = gpd.GeoDataFrame(pd.concat([plz_gdf, missing_plz_gdf], ignore_index=True), crs='EPSG:4326')

#### Case of NUTS3 regions 
For the data with NUTS3 region, Voronoi polygons are used to spatially aggregate and allocate demand more accurately. 

- Load a NUTS3 shape file for Germany t obtain the geometries of the administrative regions
- Compute centroids for each NUTS3 region that has demand
- Create Voronoi polygons around these regions to divide space based on proximity to each region center
- Calculate the area of each Voronoi polygon to assign area based weights for dmand allocation
- Allocate total demand proportionnally to each polygon based on its area within the region


In [19]:
# File path for the shapefile from Eurostat 
shapefile_dir = os.path.abspath('../../01_data/01_input_data/01_raw/NUTS_eurostat')
shapefile_base = 'NUTS_RG_60M_2021_4326.shp'
shapefile_path = os.path.join(shapefile_dir, shapefile_base)

nuts3_gdf = gpd.read_file(shapefile_path)

# Filter for German NUTS region
german_nuts3_gdf = nuts3_gdf[nuts3_gdf['NUTS_ID'].str.startswith('DE')]

# Display the filtered GeoDataFrame to see the columns structure 
print("German NUTS3 regions:")
print(german_nuts3_gdf.head())

German NUTS3 regions:
  NUTS_ID  LEVL_CODE CNTR_CODE                     NAME_LATN  \
0   DE149          3        DE                   Sigmaringen   
1   DE211          3        DE  Ingolstadt, Kreisfreie Stadt   
2   DE212          3        DE     München, Kreisfreie Stadt   
3   DE213          3        DE   Rosenheim, Kreisfreie Stadt   
4   DE214          3        DE                     Altötting   

                      NUTS_NAME  MOUNT_TYPE  URBN_TYPE  COAST_TYPE    FID  \
0                   Sigmaringen         4.0          3           3  DE149   
1  Ingolstadt, Kreisfreie Stadt         4.0          2           3  DE211   
2     München, Kreisfreie Stadt         4.0          1           3  DE212   
3   Rosenheim, Kreisfreie Stadt         4.0          2           3  DE213   
4                     Altötting         4.0          2           3  DE214   

                                            geometry  
0  POLYGON ((9.3476 48.2395, 9.6049 48.0023, 9.39...  
1  POLYGON ((11.4893

#### Aggregate demand and create network

- Call necessary sectors sheet
- Run through each sector and find closest nodes to aggregate peak load demand 
- Check the validity of demand aggregatin 
- Visualize on the map 

Steps for future improvement of the code (faster implementation)
- Instead of reading each sheets, compute if similar geometry are given, thus store information
- Think of computing the geometry of nodes in topo fie once in each case
- Avoid unecessary loops and rethink the active sheet reading
- After code is improved, add the storage and supply and save the final output 

In [20]:
# AggregateDemand function requires 'ID' column name
nodes_gdf.rename(columns={'node_id': 'ID'}, inplace=True)

# Use defined sectors to test
sheet_flags = {
    'Ind_Input': Industry,
    'Refineries_Input': Refineries,
    'Chemicals_Input': Chemicals,
    'Steel_Input': Steel,
    'Paper_Input': Paper,
    'Mineral_Processing_Input': Mineral_Processing,
    'Metal_Processing_Input': Metal_Processing,
    'Non_Metallic_Minerals_Input': Non_Metallic_Minerals,
    'Other_Industry_Input': Other_Industry,
    'Residential_Input': Residential_Heat,
    'District_Input': District_Heat,          
    'Passenger_Input': Passenger,
    'Public_Input': Public,
    'Air_Input': Air,
    'Train_Input': Train,
    'Trucks_Input': Trucks
}

# Take the ones with True as boolean value
active_sheets = [s for s, flag in sheet_flags.items() if flag]
active_sheets

NameError: name 'Industry' is not defined

In [ ]:
all_points = []

for sheet in active_sheets:
    try:
        gdf = read_sector(full_input_path, sheet, plz_gdf, german_nuts3_gdf)

        # Ensure CRS is correct
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        if not all(gdf.geometry.geom_type == 'Point'):
            gdf['geometry'] = gdf['geometry'].centroid
            gdf = gdf.set_geometry('geometry')

        # Extract latitude and longitude
        gdf['Latitude'] = gdf.geometry.y
        gdf['Longitude'] = gdf.geometry.x

        # Find closest node
        gdf['Closest_node'], gdf['Distance_km'] = find_closest_location(gdf, nodes_gdf)
        #print(gdf[['Closest_node', 'Distance_km']].head())
        all_points.append(gdf[['Closest_node', 'Peak Load [MWh/h]']])

        # Aggregate the demand
        nodes_gdf = AggregatedDemand(nodes_gdf, gdf)

    # Show possible errors
    except ValueError as e:
        print(f"Could not read sheet '{sheet}': {e}")
    except Exception as e:
        print(f"Error while processing '{sheet}': {e}")

100%|██████████| 1021/1021 [00:19<00:00, 52.42it/s]


### Check and Visualize results 

In [ ]:
# Combine all demand points
all_points = pd.concat(all_points, ignore_index=True)

# Calculate total demand per node from raw points
raw_demand_per_node = all_points.groupby('Closest_node')['Peak Load [MWh/h]'].sum().reset_index()
raw_demand_per_node.rename(columns={'Closest_node': 'ID', 'Peak Load [MWh/h]': 'Raw'}, inplace=True)

# Merge with the aggregated demand 
comparison = nodes_gdf[['ID', 'Demand']].merge(raw_demand_per_node, on='ID', how='left')
comparison['Raw'] = comparison['Raw'].fillna(0)
comparison['Difference'] = comparison['Demand'] - comparison['Raw']

# Show differences
mismatches = comparison[comparison['Difference'].abs() > 1e-3]
print(f"Number of mismatch: {len(mismatches)}")
print(mismatches.head(20))

Number of mismatch: 0
Empty DataFrame
Columns: [ID, Demand, Raw, Difference]
Index: []


In [ ]:
nodes_gdf

,ID,geometry,Longitude,Latitude,Supply,Demand
0,node_0001,POINT (6.80857 54.19578),6.808570,54.195780,0,6
1,node_0002,POINT (7.25666 53.94168),7.256660,53.941680,0,0
2,node_0003,POINT (6.12347 51.80289),6.123468,51.802895,0,0
3,node_0004,POINT (5.72234 51.84537),5.722341,51.845368,0,0
4,node_0005,POINT (6.12728 51.86375),6.127278,51.863752,0,0
...,...,...,...,...,...,...
1016,node_0655,POINT (12.94595 49.76038),12.945950,49.760375,0,7
1017,node_0656,POINT (14.34226 53.8271),14.342265,53.827100,0,0
1018,node_0657,POINT (6.38615 52.47225),6.386150,52.472250,0,3
1019,node_0658,POINT (2.46907 58.18814),2.469066,58.188137,0,0


In [ ]:
# Check non-zero demand nodes
non_zero_demand_nodes = nodes_gdf[nodes_gdf['Demand'] > 0]
print(f"Number of nodes: {len(nodes_gdf)}")
print(f"Number of nodes with non-zero demand: {len(non_zero_demand_nodes)}")

# Summarize the demand distribution
demand_summary = nodes_gdf['Demand'].describe()
print("\nStatistics:")
print(demand_summary)

Number of nodes: 1021
Number of nodes with non-zero demand: 605

Statistics:
count    1021.000000
mean       68.852106
std       232.317056
min         0.000000
25%         0.000000
50%         3.000000
75%        36.000000
max      2295.000000
Name: Demand, dtype: float64


In [ ]:
if Visualisation:
    import plotly.graph_objects as go
    import numpy as np

    df_plot = pd.DataFrame({
        'ID': nodes_gdf['ID'],
        'Longitude': nodes_gdf['Longitude'],
        'Latitude': nodes_gdf['Latitude'],
        'Demand': nodes_gdf['Demand']
    })

    # need to scale the demand for the points size on map 
    df_plot['ScaledDemand'] = df_plot['Demand'].apply(lambda d: np.log1p(d)) + 0.1
    df_plot['Used'] = df_plot['Demand'] > 0


    # Plot the map to visualize
    fig = go.Figure()

    for _, row in df_plot.iterrows():
        fig.add_trace(go.Scattergeo(
            lon=[row['Longitude']],
            lat=[row['Latitude']],
            mode='markers',
            marker=dict(
                size=max(row['ScaledDemand']*5, 2),
                color='red' if row['Used'] else 'blue',
                line=dict(width=0.5, color='black'),
                opacity=0.7
            ),
            hovertext=f"ID: {row['ID']}<br>Demand: {row['Demand']:.2f}",
            showlegend=False
        ))

    fig.update_layout(
        geo=dict(
            scope='europe',
            projection_type='natural earth',
            center=dict(lat=51.2, lon=10.4),
            lataxis=dict(range=[47, 55]),
            lonaxis=dict(range=[5, 16]),
            resolution=50
        ),
        margin={"r":0,"t":50,"l":0,"b":0},
        width=800,
        height=700,
    )

    fig.show()
